# Data Preprocessing

The objective of this notebook is to prepare the dataset for machine learning by:

- Separating features and target
- Splitting the dataset into training and testing sets
- Handling missing values
- Identifying numerical and categorical features
- Encoding categorical variables
- Scaling numerical variables
- Building a reusable preprocessing pipeline

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import joblib

* Lat Long is duplicate of Latitude and Longitude
* CLTV is a possible data leakage feature

## Load Dataset

In [16]:
df = pd.read_csv("customer.csv")

## Correct Data Types

In [17]:
df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

In [19]:
df.dtypes

CustomerID            object
Count                  int64
Country               object
State                 object
City                  object
Zip Code               int64
Lat Long              object
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges        float64
Churn Label           object
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason          object
dtype: object

## Remove Unnecessary Columns

In [20]:
columns_to_drop = [
    "CustomerID",
    "Count",
    "Churn Label",
    "Churn Score",
    "Churn Reason",
    "CLTV",
    "Lat Long"
]

df = df.drop(columns=columns_to_drop)

 <!-- Column        Reason                            
 CustomerID    Unique identifier                 
 Count         Constant column                   
 Churn Label   Duplicate target                  
 Churn Score   Target leakage                    
 Churn Reason  Available only after churn        
 CLTV          Possible target leakage           
 Lat Long      Duplicate of Latitude & Longitude  -->


In [21]:
df

,Country,State,City,Zip Code,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,United States,California,Los Angeles,90003,33.964131,-118.272783,Male,No,No,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,United States,California,Los Angeles,90005,34.059281,-118.307420,Female,No,No,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,United States,California,Los Angeles,90006,34.048013,-118.293953,Female,No,No,Yes,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
3,United States,California,Los Angeles,90010,34.062125,-118.315709,Female,No,Yes,Yes,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,United States,California,Los Angeles,90015,34.039224,-118.266293,Male,No,No,Yes,...,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,United States,California,Landers,92285,34.341737,-116.539416,Female,No,No,No,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),21.15,1419.40,0
7039,United States,California,Adelanto,92301,34.667815,-117.536183,Male,No,Yes,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,0
7040,United States,California,Amboy,92304,34.559882,-115.637164,Female,No,Yes,Yes,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,0
7041,United States,California,Angelus Oaks,92305,34.167800,-116.864330,Female,No,Yes,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,0


## Separate Features and Target

In [22]:
X = df.drop(columns=["Churn Value"])

y = df["Churn Value"]

## Train-Test Split

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [50]:
print("Training Target Distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting Target Distribution:")
print(y_test.value_counts(normalize=True))

Training Target Distribution:
Churn Value
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Testing Target Distribution:
Churn Value
0    0.734564
1    0.265436
Name: proportion, dtype: float64


## Identify Numerical & Categorical Features

In [24]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print(numerical_features)
print(categorical_features)

['Zip Code', 'Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Total Charges']
['Country', 'State', 'City', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


In [25]:
X_train.isnull().sum()

Country              0
State                0
City                 0
Zip Code             0
Latitude             0
Longitude            0
Gender               0
Senior Citizen       0
Partner              0
Dependents           0
Tenure Months        0
Phone Service        0
Multiple Lines       0
Internet Service     0
Online Security      0
Online Backup        0
Device Protection    0
Tech Support         0
Streaming TV         0
Streaming Movies     0
Contract             0
Paperless Billing    0
Payment Method       0
Monthly Charges      0
Total Charges        8
dtype: int64

## Build Numerical Pipeline

In [26]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## Build Categorical Pipeline

In [27]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

## Combine Pipelines

In [29]:
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

## Fit on training and test dataset

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [35]:
print("Training Shape :", X_train_processed.shape)
print("Testing Shape  :", X_test_processed.shape)

Training Shape : (5634, 1179)
Testing Shape  : (1409, 1179)


In [36]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)

['num__Zip Code' 'num__Latitude' 'num__Longitude' ...
 'cat__Payment Method_Credit card (automatic)'
 'cat__Payment Method_Electronic check' 'cat__Payment Method_Mailed check']


## Convert to DataFrame

In [38]:
X_train_processed = pd.DataFrame(
    X_train_processed.toarray(), 
    columns=feature_names, 
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed.toarray(), 
    columns=feature_names, 
    index=X_test.index
)

In [40]:
X_train_processed.head()

,num__Zip Code,num__Latitude,num__Longitude,num__Tenure Months,num__Monthly Charges,num__Total Charges,cat__Country_United States,cat__State_California,cat__City_Acampo,cat__City_Acton,...,cat__Streaming Movies_Yes,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__Paperless Billing_No,cat__Paperless Billing_Yes,cat__Payment Method_Bank transfer (automatic),cat__Payment Method_Credit card (automatic),cat__Payment Method_Electronic check,cat__Payment Method_Mailed check
4626,-0.641938,-0.748174,1.205900,0.102371,-0.521976,-0.263290,1.0,1.0,0.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4192,1.302937,1.352382,-1.041750,-0.711743,0.337478,-0.504815,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
5457,1.345205,1.652222,-1.234760,-0.793155,-0.809013,-0.751214,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
4717,-0.441297,-1.021180,0.830735,-0.263980,0.284384,-0.173700,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
4673,-0.515133,-1.028350,1.270977,-1.281624,-0.676279,-0.990851,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [47]:
X_train_processed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5634 entries, 4626 to 6017
Columns: 1179 entries, num__Zip Code to cat__Payment Method_Mailed check
dtypes: float64(1179)
memory usage: 50.7 MB


## Save the Preprocessor

In [41]:
joblib.dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']

## Save Processed Data

In [42]:
X_train_processed.to_csv("X_train_processed.csv", index=False)
X_test_processed.to_csv("X_test_processed.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

In [48]:
print(numerical_features)

['Zip Code', 'Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Total Charges']


In [49]:
numerical_features.remove("Zip Code")
categorical_features.append("Zip Code")

## Summary

The preprocessing pipeline performs the following tasks:

- Corrected data types
- Removed identifier and leakage features
- Split the data into training and testing sets
- Imputed missing values
- Scaled numerical features
- One-hot encoded categorical features
- Created reusable preprocessing pipelines
- Saved the preprocessor for future inference